<a href="https://colab.research.google.com/github/smithblaine014-cyber/Test1/blob/main/irrigationCode.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install streamlit pyngrok pandas matplotlib

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 49.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 52.4 MB/s eta 0:00:00


In [10]:
%%writefile irrigation_app.py

import streamlit as st
import pandas as pd
import matplotlib.pyplot as plt

st.title("🌱 Irrigation Scheduling App")

st.write("""
Upload a weather dataset containing:

Month
Date
Year
ET_inches
Precipitation_inches
""")

# Upload file inside the app
uploaded_file = st.file_uploader("Upload Weather CSV", type=["csv"])

# Sidebar settings
st.sidebar.header("Field Settings")

kc = st.sidebar.number_input("Crop Coefficient (Kc)", value=1.15)

soil_awc = st.sidebar.number_input(
    "Soil Available Water Capacity (in/ft)",
    value=2.0
)

root_depth = st.sidebar.number_input(
    "Root Depth (ft)",
    value=3.0
)

allowed_depletion = st.sidebar.slider(
    "Allowed Depletion (%)",
    10,
    80,
    50
)

# Run app only if file uploaded
if uploaded_file is not None:

    df = pd.read_csv(uploaded_file)

    # Build real date column
    df["Date_full"] = pd.to_datetime({
        "year": df["Year"],
        "month": df["Month"],
        "day": df["Date"]
    })

    st.subheader("Preview of Uploaded Data")
    st.write(df.head())

    # Extract weather variables
    eto = df["ET_inches"]
    rain = df["Precipitation_inches"]

    # Crop ET
    etc = eto * kc

    # Soil water calculations
    taw = soil_awc * root_depth
    mad = taw * (allowed_depletion / 100)

    soil_water = taw
    depletion = []

    for i in range(len(df)):

        soil_water = soil_water - etc.iloc[i] + rain.iloc[i]

        if soil_water > taw:
            soil_water = taw

        depletion.append(taw - soil_water)

    df["ETc"] = etc
    df["Depletion"] = depletion
    df["Irrigation_Needed"] = df["Depletion"] > mad

    st.subheader("Calculated Irrigation Data")
    st.write(df)

    # GRAPH 1 — Soil Water Depletion
    st.subheader("Soil Water Depletion")

    fig1, ax1 = plt.subplots()

    ax1.plot(df["Date_full"], df["Depletion"], label="Soil Water Depletion")
    ax1.axhline(mad, linestyle="--", label="Max Allowable Depletion")

    ax1.set_xlabel("Date")
    ax1.set_ylabel("Inches")
    ax1.legend()

    st.pyplot(fig1)

    # GRAPH 2 — ET and Rain
    st.subheader("Daily Water Balance")

    fig2, ax2 = plt.subplots()

    ax2.plot(df["Date_full"], eto, label="Reference ET")
    ax2.plot(df["Date_full"], etc, label="Crop ET")
    ax2.bar(df["Date_full"], rain, label="Rainfall")

    ax2.set_xlabel("Date")
    ax2.set_ylabel("Inches")
    ax2.legend()

    st.pyplot(fig2)

    # Irrigation recommendation
    st.subheader("Recommended Irrigation Days")

    irrigation_days = df[df["Irrigation_Needed"]]

    st.write(irrigation_days)

else:

    st.info("Upload a CSV file to start the irrigation scheduler.")

Overwriting irrigation_app.py


In [11]:
from pyngrok import ngrok
import time

!ngrok config add-authtoken 3AlSpu1VEA5352mHHbhnyISNvMw_6EsfFeiBzeaCuHmbSaA3Z

get_ipython().system_raw('streamlit run irrigation_app.py --server.port 8501 &')

time.sleep(3)

public_url = ngrok.connect(8501)

print("Open the app here:")
print(public_url)

Authtoken saved to configuration file: /root/.config/ngrok/ngrok.yml
Open the app here:
NgrokTunnel: "https://unbrined-alfonzo-incandescently.ngrok-free.dev" -> "http://localhost:8501"
